# LC25000: Avoiding augmentation-induced data leakage

The **LC25000** dataset ([Borkowski et al., 2019](https://arxiv.org/abs/1912.12142v1)) is one of the most widely used histopathology benchmarks, with over 650 citing papers (as of June 2026). It contains 25,000 lung and colon tissue image patches across 5 classes.

## ⚠️ The problem: augmentation-induced data leakage

The 25,000 images were not collected independently. Instead, **250 original tissue tiles per class (1,250 total) were augmented** with random rotations (up to 25°) and horizontal/vertical flips to produce ~20 correlated copies per tile.

If you split the dataset **randomly** — augmented copies of the same original tile end up in **both** your training and test sets. This is a form of data leakage that **inflates accuracy** and does not reflect true generalisation.

We show below that:
- The **Train/Test split provided in this Kaggle upload** already leaks (~98% of tile-groups are contaminated).
- Any naive random split does the same.
- A **group-aware split** (splitting on tile-groups, not individual images) eliminates the leak entirely.

## ✅ The solution: group-aware splitting

This notebook loads pre-computed **tile-group annotations** that assign each of the 25,000 images to its original tile group (`group_id`). These annotations were produced by a semi-automatic pipeline (foundation model feature extraction → KMeans clustering → manual correction) described in:

> Batchkala, G., Li, B., Rittscher, J. *Evaluating Histopathology Foundation Models for Few-Shot Tissue Clustering: An Application to LC25000 Augmented Dataset Cleaning.* DEMI @ MICCAI 2024. **Best Paper Award.**
> DOI: [10.1007/978-3-031-73748-0_2](https://doi.org/10.1007/978-3-031-73748-0_2) | [GitHub](https://github.com/GeorgeBatch/LC25000-clean)

```bibtex
@inproceedings{batchkala2025EvaluatingHistopathologyFoundation,
  title     = {Evaluating {Histopathology Foundation Models} for~{Few-Shot Tissue Clustering}:
               {An~Application} to~{LC25000 Augmented Dataset Cleaning}},
  author    = {Batchkala, George and Li, Bin and Rittscher, Jens},
  booktitle = {Data Engineering in Medical Imaging},
  pages     = {11--21},
  year      = {2025},
  publisher = {Springer Nature Switzerland},
  doi       = {10.1007/978-3-031-73748-0_2},
}
```

---
## Imports and constants

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold

In [ ]:
# ── Constants (inlined from source/constants.py — self-contained on Kaggle) ──
RANDOM_SEED          = 42
ALL_CANCER_TYPES     = ('colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc')
NUM_TOTAL_IMAGES     = 25_000
NUM_CLASS_IMAGES     = 5_000
NUM_CLASS_PROTOTYPES = 250   # original tiles per class before augmentation

# ── Kaggle input paths ───────────────────────────────────────────────────────
GROUPS_CSV  = '/kaggle/input/lc25000-clean-groups/lc25000_image_groups.csv'
IMAGES_ROOT = '/kaggle/input/datasets/javaidahmadwani/lc25000/lung_colon_image_set'

---
## 1. Load the tile-group annotations

In [ ]:
groups_df = pd.read_csv(GROUPS_CSV)
print(f"Shape            : {groups_df.shape}")
print(f"Unique group_ids : {groups_df['group_id'].nunique()}")
print(f"Unique labels    : {sorted(groups_df['label'].unique())}")
groups_df.head()

---
## 2. Attach image paths and identify the dataset's provided split

The `javaidahmadwani/lc25000` dataset already comes with a pre-made split into
`Train and Validation Set/` and `Test Set/`.
We match every image file to our tile-group annotation by **filename stem**
(the part before the extension), which is invariant to extension (`.jpg`/`.jpeg`) and folder layout.

In [ ]:
# Recursively find all images under the dataset root
image_files = [
    p for p in glob.glob(os.path.join(IMAGES_ROOT, '**', '*'), recursive=True)
    if os.path.isfile(p) and p.lower().endswith(('.jpg', '.jpeg', '.png'))
]
print(f"Image files found: {len(image_files)}")

# Build stem → path and stem → split-name mappings
stem_to_path  = {}
stem_to_split = {}
for p in image_files:
    stem = Path(p).stem
    stem_to_path[stem] = p
    if 'Test Set' in p:
        stem_to_split[stem] = 'Test Set'
    elif 'Train' in p:
        stem_to_split[stem] = 'Train and Validation Set'
    else:
        stem_to_split[stem] = 'unknown'

# Merge onto the groups dataframe
groups_df['path']           = groups_df['stem'].map(stem_to_path)
groups_df['provided_split'] = groups_df['stem'].map(stem_to_split)

n_matched   = groups_df['path'].notna().sum()
n_unmatched = groups_df['path'].isna().sum()
print(f"Matched   : {n_matched} / {NUM_TOTAL_IMAGES}")
if n_unmatched > 0:
    warnings.warn(
        f"{n_unmatched} images could not be matched to a file path. "
        "Check that IMAGES_ROOT points to the correct dataset directory."
    )

print("\nImages per provided split:")
print(groups_df['provided_split'].value_counts())

---
## 3. Quantify the data leakage

### 3a. The Train/Test split in this Kaggle upload already leaks

This Kaggle version of LC25000 comes with images pre-divided into
`Train and Validation Set/` and `Test Set/` folders. Since the original
dataset is just a flat collection of augmented images with no intended split,
this division was made arbitrarily — and as we show below, it leaks tile-groups
across the boundary.

In [ ]:
train_groups = set(groups_df.loc[groups_df['provided_split'] == 'Train and Validation Set', 'group_id'])
test_groups  = set(groups_df.loc[groups_df['provided_split'] == 'Test Set',                  'group_id'])

leaked_groups = train_groups & test_groups
total_groups  = train_groups | test_groups

print(f"Total unique tile-groups : {len(total_groups)}")
print(f"Groups in train+val      : {len(train_groups)}")
print(f"Groups in test           : {len(test_groups)}")
print(f"Groups in BOTH (leaked)  : {len(leaked_groups)}")

if len(total_groups) == 0:
    print("\nNo images matched to the provided split.")
    print("Make sure the javaidahmadwani/lc25000 dataset is attached as an input to this notebook.")
else:
    print(f"Contamination rate       : {100 * len(leaked_groups) / len(total_groups):.1f}%")
    test_df = groups_df[groups_df['provided_split'] == 'Test Set']
    frac_leaked = test_df['group_id'].isin(train_groups).mean()
    print(f"\nFraction of test images whose tile-group is also in training: {100 * frac_leaked:.1f}%")

### 3b. Why any random split leaks: a Monte-Carlo demonstration

Even if you ignore the provided split and re-split randomly, the contamination is essentially the same.
This reproduces the calculation from the companion repository's notebook `0-expected-data-leakage-calculation.ipynb`.

In [ ]:
n_clusters    = NUM_CLASS_PROTOTYPES  # 250 per class
n_total       = NUM_CLASS_IMAGES      # 5000 per class
test_size     = 0.2
n_experiments = 1000

rng = np.random.default_rng(RANDOM_SEED)

proportions_random  = []
proportions_grouped = []

for _ in range(n_experiments):
    # Simulate ~20 images per group (matches LC25000's augmentation structure)
    cluster_assignments = rng.integers(0, n_clusters, size=n_total)

    # Leaky: random stratified split (ignores groups)
    tr_idx, te_idx = train_test_split(
        np.arange(n_total), test_size=test_size,
        stratify=cluster_assignments, random_state=None,
    )
    n_contaminated = len(set(cluster_assignments[tr_idx]) & set(cluster_assignments[te_idx]))
    proportions_random.append(n_contaminated / n_clusters)

    # Clean: group-aware split
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=None)
    tr_g, te_g = next(gss.split(np.arange(n_total), groups=cluster_assignments))
    n_contaminated_g = len(set(cluster_assignments[tr_g]) & set(cluster_assignments[te_g]))
    proportions_grouped.append(n_contaminated_g / n_clusters)

print(f"Random split      — mean contamination: {100*np.mean(proportions_random):.1f}% "
      f"± {100*np.std(proportions_random):.1f}%")
print(f"Group-aware split — mean contamination: {100*np.mean(proportions_grouped):.1f}% "
      f"± {100*np.std(proportions_grouped):.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(proportions_random,  bins=20, alpha=0.7, color='tomato',    label='Random / stratified split')
ax.hist(proportions_grouped, bins=20, alpha=0.7, color='steelblue', label='Group-aware split')
ax.set_xlabel('Fraction of tile-groups contaminated', fontsize=12)
ax.set_ylabel('Count (over 1,000 simulations)', fontsize=12)
ax.set_title('Tile-group contamination rate per class\n(250 groups, 5,000 images, 80/20 split)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

---
## 4. Reusable helpers for group-aware splitting

Use these in your own training code.
The key requirement: **all images from the same `group_id` must stay on the same side of every split.**

In [ ]:
def grouped_train_test_split(df: pd.DataFrame,
                              test_size: float = 0.2,
                              seed: int = RANDOM_SEED):
    """Split a DataFrame so that no tile-group appears on both sides."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(gss.split(df, groups=df['group_id']))
    train_df = df.iloc[train_idx].copy()
    test_df  = df.iloc[test_idx].copy()
    leaked = set(train_df['group_id']) & set(test_df['group_id'])
    assert len(leaked) == 0, f"Group-aware split still has {len(leaked)} leaked groups!"
    return train_df, test_df


train_df, test_df = grouped_train_test_split(groups_df)
leaked = set(train_df['group_id']) & set(test_df['group_id'])
print(f"Train: {len(train_df)} images | Test: {len(test_df)} images | Leaked groups: {len(leaked)}")
print("\nLabel distribution in train:")
print(train_df['label'].value_counts())

In [ ]:
# Cross-validation example with StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5)
print("StratifiedGroupKFold — fold sizes and leaked groups:")
for fold_i, (tr_idx, val_idx) in enumerate(
        sgkf.split(groups_df, y=groups_df['label'], groups=groups_df['group_id'])):
    fold_train = groups_df.iloc[tr_idx]
    fold_val   = groups_df.iloc[val_idx]
    leaked_fold = set(fold_train['group_id']) & set(fold_val['group_id'])
    print(f"  Fold {fold_i+1}: train={len(fold_train)}, val={len(fold_val)}, leaked groups={len(leaked_fold)}")

---
## 5. Classification demo: accuracy inflation in practice

We extract ResNet18 (ImageNet-pretrained) features from a subset of the images,
then compare classification accuracy under three splitting strategies:

| Split | Leaky? |
|---|---|
| (a) Train/Test folders in this Kaggle upload | ✅ Yes — ~98%+ of groups contaminated |
| (b) Random/stratified re-split | ✅ Yes — ~98%+ of groups contaminated |
| (c) Group-aware re-split (`GroupShuffleSplit`) | ❌ No — 0% contamination |

We expect (a) and (b) to overestimate accuracy compared to the honest (c).

> **Runtime note:** Reduce `N_PER_CLASS` for a quick CPU run.
> Set `N_PER_CLASS = None` to use all 5,000 images per class on GPU (recommended).

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Configuration ─────────────────────────────────────────────────────────────
N_PER_CLASS = None   # use all 5,000 images per class; set to e.g. 500 for a quick CPU test
BATCH_SIZE  = 128
N_SPLITS    = 5      # repeated random/grouped splits to get mean ± std
TEST_SIZE   = 0.2

# Safe device detection — some assigned GPUs may not be compatible with
# the installed PyTorch build; fall back to CPU if a test operation fails
def _get_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            return 'cuda'
        except Exception:
            pass
    return 'cpu'

DEVICE = _get_device()
print(f"Device: {DEVICE}")

# Drop rows with missing paths
df_images = groups_df.dropna(subset=['path']).copy()

if N_PER_CLASS is not None:
    df_images = (
        df_images
        .groupby('label', group_keys=False)
        .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )

print(f"Using {len(df_images)} images: {df_images['label'].value_counts().to_dict()}")

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


imagenet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ResNet18 with the classification head removed → 512-d feature vector
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor = feature_extractor.to(DEVICE).eval()

dataset    = ImagePathDataset(df_images['path'].tolist(), imagenet_transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=(DEVICE == 'cuda'))

features_list = []
with torch.no_grad():
    for batch in dataloader:
        feats = feature_extractor(batch.to(DEVICE))   # (B, 512, 1, 1)
        features_list.append(feats.squeeze(-1).squeeze(-1).cpu().numpy())

X = np.concatenate(features_list, axis=0)
print(f"Feature matrix shape: {X.shape}")

In [ ]:
le = LabelEncoder()
y      = le.fit_transform(df_images['label'].values)
groups = df_images['group_id'].values
provided = df_images['provided_split'].values


def evaluate_split(X_tr, y_tr, X_te, y_te):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_SEED)
    clf.fit(X_tr_s, y_tr)
    return accuracy_score(y_te, clf.predict(X_te_s))


accs_provided = []
accs_random   = []
accs_grouped  = []

for seed in range(N_SPLITS):
    # (a) Dataset's own provided split — single split, evaluated once
    if seed == 0:
        train_mask = (provided == 'Train and Validation Set')
        test_mask  = (provided == 'Test Set')
        if train_mask.any() and test_mask.any():
            accs_provided.append(
                evaluate_split(X[train_mask], y[train_mask], X[test_mask], y[test_mask])
            )

    # (b) Leaky random/stratified split
    tr_idx, te_idx = train_test_split(
        np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    accs_random.append(evaluate_split(X[tr_idx], y[tr_idx], X[te_idx], y[te_idx]))

    # (c) Group-aware split
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    tr_g, te_g = next(gss.split(X, y, groups=groups))
    accs_grouped.append(evaluate_split(X[tr_g], y[tr_g], X[te_g], y[te_g]))

print("Classification accuracy (ResNet18-ImageNet, Logistic Regression):\n")
if accs_provided:
    print(f"  (a) Provided Train/Test split : {100*np.mean(accs_provided):.1f}%")
print(    f"  (b) Random/stratified split  : {100*np.mean(accs_random):.1f}% ± {100*np.std(accs_random):.1f}%  (n={N_SPLITS})")
print(    f"  (c) Group-aware split        : {100*np.mean(accs_grouped):.1f}% ± {100*np.std(accs_grouped):.1f}%  (n={N_SPLITS})")

In [ ]:
labels_plot = []
means_plot  = []
stds_plot   = []

if accs_provided:
    labels_plot.append('(a) Kaggle upload split\n(leaky)')
    means_plot.append(np.mean(accs_provided))
    stds_plot.append(0.0)
labels_plot += ['(b) Random split\n(leaky)', '(c) Group-aware split\n(honest)']
means_plot  += [np.mean(accs_random), np.mean(accs_grouped)]
stds_plot   += [np.std(accs_random),  np.std(accs_grouped)]

colors = ['tomato'] * (len(labels_plot) - 1) + ['steelblue']
x_pos  = np.arange(len(labels_plot))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(x_pos, [m * 100 for m in means_plot],
              yerr=[s * 100 for s in stds_plot],
              color=colors, capsize=6, width=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels_plot, fontsize=11)
ax.set_ylabel('Test accuracy (%)', fontsize=12)
ax.set_title('Leaky vs honest splits: classification accuracy\n(ResNet18-ImageNet, Logistic Regression)', fontsize=11)
ax.set_ylim(50, 102)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for bar, mean in zip(bars, means_plot):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{100*mean:.1f}%', ha='center', va='bottom', fontsize=11)
fig.tight_layout()
plt.show()

---
## 6. How to use this in your own project

```python
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the group annotations (attach the lc25000-clean-groups dataset to your notebook)
groups_df = pd.read_csv('/kaggle/input/lc25000-clean-groups/lc25000_image_groups.csv')

# ... build your own DataFrame with 'group_id' merged in ...

# Group-aware 80/20 split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['group_id']))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

# Verify: zero groups appear on both sides
assert len(set(train_df['group_id']) & set(test_df['group_id'])) == 0
```

---

## Citation

If you use these annotations in your work, please cite:

```bibtex
@inproceedings{batchkala2025EvaluatingHistopathologyFoundation,
  title     = {Evaluating {Histopathology Foundation Models} for~{Few-Shot Tissue Clustering}:
               {An~Application} to~{LC25000 Augmented Dataset Cleaning}},
  author    = {Batchkala, George and Li, Bin and Rittscher, Jens},
  booktitle = {Data Engineering in Medical Imaging},
  pages     = {11--21},
  year      = {2025},
  publisher = {Springer Nature Switzerland},
  doi       = {10.1007/978-3-031-73748-0_2},
}
```

GitHub: https://github.com/GeorgeBatch/LC25000-clean
Paper: https://doi.org/10.1007/978-3-031-73748-0_2